Lets first review the bronze table to make sure they are intact

In [0]:
# Set the current catalog
spark.catalog.setCurrentCatalog("workspace")

# Set the current schema (database)
spark.catalog.setCurrentDatabase("genai_course")


In [0]:
bronze_df = spark.read.table("bronze_docs_raw");
display(bronze_df)

Silver layer represents a refined copy of bronze layer. We use ai_parse_document to process unstructured documents and extract their content in a JSON formatted structure. This uses LLM

In [0]:
%sql

select path, ai_parse_document(content) as parsed_content from bronze_docs_raw;

For additional metadata such as keys, brackets etc use map version 2.0. This is advanced.

In [0]:
%sql
select path, ai_parse_document(content, map('version', '2.0')) as parsed_content from bronze_docs_raw;

LLM can be used to convert JSON data into clean markdown formatted text which preserves document semantics such as headers, tables and captions. In DBX, we can easily call an LLM on a batch of data using the ai_query function.

In [0]:
%sql

WITH parsed_docs as  (
    select path, ai_parse_document(content) as parsed_content from bronze_docs_raw
)
SELECT path, parsed_content, ai_query(
    'databricks-gpt-oss-20b',
    concat ('Given a JSON object respresenting a parsed document (with pages, elements and metadata), convert the content into clean, readable markdown. Use "==page==" to separate each page without adding page number. Preserve important structures such as pages, headers, tables and captions. Do not include any code blocks or JSON in the output just the clean markdown text. JSON:',
    CAST(parsed_content AS STRING))
) FROM parsed_docs

We can call SQL functions in Python using the expression function from PySpark. This function executes SQL-Like expressions and returns a column object. We create a new dataframe prase_df from bronze_df and add new columns parsed_content and plain_text by using the expression method and executing SQL

In [0]:
from pyspark.sql.functions import expr

endpoint = "databricks-gpt-oss-20b"

prompt_prefix = ''' 
Given a JSON object respresenting a parsed document (with pages, elements and metadata), convert the content into clean, readable markdown. Use "==page==" to separate each page without adding page number. Preserve important structures such as pages, headers, tables and captions. Do not include any code blocks or JSON in the output just the clean markdown text. 

JSON:
'''

parsed_df = (bronze_df.withColumn("parsed_content", 
                                  expr(f"""ai_parse_document(content, 
                                       map('version', '2.0', 'imageOutputPath', '/Volumes/workspace/genai_course/dataset/imgs'))"""))
             .withColumn("plain_text",
                         expr(f"""ai_query('{endpoint}', concat('{prompt_prefix}', CAST(parsed_content as string)), responseFormat => '{{"type": "text"}}')"""))
             .select("path", "parsed_content", "plain_text")
            )

display(parsed_df)

Save as a Silver Layer Table

In [0]:
parsed_df.write.mode("overwrite").saveAsTable("Silver_docs_parsed")

Using Databricks helper function render_ai_parse_output function to Visualize the parsed outputs. Copy the code from databricks site and place it in includes folder as document_render.py

In [0]:
import sys, os
import importlib

sys.path.append("/Workspace/Users/ssagi@me.com/DBX_GenAI_Certification/Includes")
importlib.invalidate_caches()

from document_renderer import render_ai_parse_output, render_ai_parse_output_interactive

print("document_renderer imported successfully")


Proceed with Visualization

In [0]:
sample = parsed_df.select("parsed_content").limit(1).collect()
doc = sample[0]["parsed_content"]
display(render_ai_parse_output_interactive(doc))
